I am looking at the Open Parking and Camera Violations dataset from NYC Open Data, provided by the Department of Finance (last updated November 10, 2026). 

The full dataset contains 153 million rows and 19 columns, so I downloaded 50,000 rows through the API, keeping only the issue date, violation type, fine amount, county, and issuing agency. 

I will be focusing on the fine amount column (units in USD).


Data soruce: [Open Parking and Camera Violations](https://data.cityofnewyork.us/City-Government/Open-Parking-and-Camera-Violations/nc67-uf89/about_data)

In [1]:
# Data and column selection for parking violations analysis
data = "parking_violations.csv"
column = "fine_amount"

In [3]:
# Load the parking violations

import pandas as pd

violations = pd.read_csv(data)


In [4]:
# Preview the first five rows of the data

violations.head()

,issue_date,violation,fine_amount,county,issuing_agency
0,11/25/2016,DOUBLE PARKING,115.0,K,TRAFFIC
1,08/16/2019,PHTO SCHOOL ZN SPEED VIOLATION,50.0,MN,DEPARTMENT OF TRANSPORTATION
2,06/02/2017,PHTO SCHOOL ZN SPEED VIOLATION,50.0,QN,DEPARTMENT OF TRANSPORTATION
3,10/23/2016,CROSSWALK,115.0,K,TRAFFIC
4,10/23/2016,FIRE HYDRANT,115.0,K,TRAFFIC


In [ ]:
# Check the columns and types of the data

violations.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   issue_date      50000 non-null  str    
 1   violation       47509 non-null  str    
 2   fine_amount     47509 non-null  float64
 3   county          47074 non-null  str    
 4   issuing_agency  47509 non-null  str    
dtypes: float64(1), str(4)
memory usage: 1.9 MB


Data cleaning: 
Before starting the calculations, I will check my column of interst (fine_amount) for any missing values. 

Tickets that don't have a fine amount cannot be included in the calculations. 

I also want to check for "strange values" using the describe method. This will tell me whether there are any outliers that seem implausible or negative numbers that would also be implausible fine amounts. 

In [ ]:
# Check for missing values in the fine_amount column
missing_count = violations[column].isna().sum()
missing_percent = missing_count / len(violations) * 100
print(f"Missing values in {column}: {missing_count} ({missing_percent:.2f}%)")


Missing values in fine_amount: 2491 (4.98%)


In [ ]:
# Check for strange values
violations[column].describe()

count    47509.000000
mean        67.548254
std         29.711245
min         35.000000
25%         45.000000
50%         50.000000
75%         95.000000
max        515.000000
Name: fine_amount, dtype: float64

In [ ]:
# Check the violation type of the maximum fine amount to assess whether
# It is a plausible outlier or an artificially exaggerated amount.
violations[violations[column] == violations[column].max()]["violation"].value_counts()

violation
FAILURE TO DISPLAY BUS PERMIT    1
Name: count, dtype: int64

We find the cheapest fine is $35, so there are no $0 or negative fine amounts. The spread, $35 to $515 is plausible for NYC tickets, and it suggests there are no major outliers or falsely listed/exaggerated values. We see the highest value ($515) was charged due to the failure to display a bus permit. Checking this value against Violation Code 1 of the Fine Master Table, we see that it is correct. We do find there are 2491 tickets with no listed fine amount, and these will be removed. 

Data source: [Fine Master Table](https://www.nyc.gov/assets/finance/downloads/pdf/tax_and_parking_program_operations/stipulated-fines-fee-schedule.pdf)

Now, I will remove the missing values and begin the easy mean, median mode calculations. 

In [ ]:
# Remove tickets with no fine amount
fines = violations[column].dropna()
print(f"Number of tickets with fine amounts: {len(fines)}")

Number of tickets with fine amounts: 47509


In [25]:
# Calculating mean, median, and mode

mean = fines.mean()
median = fines.median()
modes = fines.mode()

print(f"Mean fine amount: ${mean:,.2f}")
print(f"Median fine amount: ${median:,.2f}")
print(f"Most common fine (Mode): ${list(modes)}")

Mean fine amount: $67.55
Median fine amount: $50.00
Most common fine (Mode): $[50.0]


The mean fine amount is $67.5, while the most common (mode) and the median fine amount are equal at $50.0. The mean is higher than the median indicating a right skew, since a smaller number of expensive tickets raise the average. 

Now I will progress on to the hard way of finding the mean, median and mode. I will avoid accessing the Data Frame/Series as well as functions with the names mean, median, mode. I will read the CSV file directly, and I will remove the tickets that contain no file amount, as before. The list I will create in my code to hold the numbers will be called fine_amounts. 

My approach to calculations is as follows: 

1. Mean: the idea is to add up every fine and then divide by how many fines there in total. I will start a running total at zero, add each fine value to this total to get the sum of all the 47,509 fines. I will then divide this total by len(fine_amounts) (the total number of entries) to get the manual mean. To do this, I will use a for loop that will allow me to iterate through the values and add them to my total. 

2. Median: the idea is to sort all the values from smallest to largest and pick the one in the middle, but the exercise differs depending on whether there are an even or odd number of entries. I will create a list of sorted fines using the sorted function; I will then count its length. I will then identify the middle by dividing the count by two using `//` (you drop the remainder). 

For odd lists: if count % 2 == 1 then the list contains an odd number of entries. In this case, the median is the middle. If I use this line of code:  `manual_median = sorted_fines[middle]` then it will use the quotient as a position (index). 

For even lists: if the list contains an even number of values, then the median is an average of the two middle values. I will still use the logic of inputting the quotient from the middle variable into "sorted_fines" but this time I will add it to the value that is just before it (at the position middle - 1). I will then divide this sum by two to get the average. 

3. Mode: the idea is to count how many times each fine amount appears and then select the amount that appears the most frequently. I will create an empty dictionary titled fine_counts in which the keys will be fine amounts and the values will be how many times each appears. I will then start a for loop `for value in fine_amounts:` to go through every fine. I will then create an if statement asking whether this amount has been seen before, and if so I will start its count at one and add to it: `if value in fine_counts:...`. If it hasn't been seen before, I will use an else statement to start its count at 1 `fine_counts[value] = 1`

I will create a new variable titled highest_count that will be the highest count found in the dictionary. Finally, I will use a list comprehension to collect every fine amount whose count equals that highest count. I will get each key-value pair together by using `fine_counts.items()` and then say `for value, count in...` to loop through these pairs and split each of them into two variables: value (fine amount) and count (its tally). Then, I say `if count == highest_count` to filter out the pair only if its count equals the top count. At the front of my list comprehension, I will use the variable `value` since this is what to keep for each pair that passes the filter (we don't want to report the tally as well). 

In [ ]:
# Data cleaning: I will only count tickets with a fine amount.
# Use csv.DictReader to read the file one row at a time and
# turn every row into a dictionary, with key being column names
# from the first line, and the values being the data.
# I will then use a for loop to iterate through each row, which is
# one ticket's dictionary. The strip method will remove any leading
# and trailing whitespace from the fine amount.
# If the fine amount is empty, I will skip that row so it is not included
# in my fine_amounts list.

import csv

fine_amounts = []
with open(data, newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        raw_value = row[column].strip()
        if raw_value == "":
            continue
        fine_amounts.append(float(raw_value))

print(f"Tickets with a fine amount: {len(fine_amounts)}")


Tickets with a fine amount:,47509


In [ ]:
# For the Mean

total = 0
for value in fine_amounts:
    total += value

manual_mean = total / len(fine_amounts)
print(f"Manual mean fine amount: ${manual_mean:,.2f}")

Manual mean fine amount: $67.55


In [ ]:
# For the Median

sorted_fines = sorted(fine_amounts)
count = len(sorted_fines)
middle = count // 2

if count % 2 == 0:
    manual_median = sorted_fines[middle]
else:
    manual_median = (sorted_fines[middle - 1] + sorted_fines[middle]) / 2

print(f"Manual median fine amount: ${manual_median:,.2f}")


Manual median fine amount: $50.00


In [ ]:
# For the Mode

fine_counts = {}
for value in fine_amounts:
    if value in fine_counts:
        fine_counts[value] += 1
    else:
        fine_counts[value] = 1

highest_count = max(fine_counts.values())

manual_modes = [value for value, count in fine_counts.items() if count == highest_count]

print(f"Manual mode fine amounts: {manual_modes}")

Manual mode fine amounts: [50.0]


Data Visualisation 

I will begin with a simple bar chart of the most common fine amounts, with each bar's length showing how many tickets had that fine. 

To build this chart, I set two settings (I) how amny fine amounts to display (tne) and (II) how long the longest bar should be (40 characters). Using pandas, I then counted how many tickets had each fine amount, kept the 10 most common, and sorted them from cheapest to most expensive. For each fine amount, the code draws a bar out of repeated █ characters. Each bar is scaled up against the most common fine, which gets the full 40 characters and the others are proportionally shorter, with the minimum values getting one block, so they stay visible. 

The chart shows us that $50 is the most common fine, but that $115 is close behind; this large group of $115 values helps us to explain why the mean is pushed up and is higher than the medium. 

In [ ]:
# How many fine amounts to display
top_n = 10
# Determine length of the longest bar, in characters
bar_max_width = 40

# Count tickets at each fine amount, keeping the n (10) most common fine amounts
# Use sort_index to sort the fines in dollar order
top_fines = fines.value_counts().head(top_n).sort_index()
most_tickets = top_fines.max()

# Print
print(f"The {top_n} most common NYC parking fine amounts")
print("Each bar shows the number of tickets\n")

for amount, count in top_fines.items():
    bar = "█" * max(1, round(count / most_tickets * bar_max_width))
    print(f"${amount:>4.0f} {bar} {count:,}")


The 10 most common NYC parking fine amounts
Each bar shows the number of tickets

$  35 ████████████████████ 5,852
$  45 ██████████████████████ 6,565
$  50 ████████████████████████████████████████ 11,676
$  55 █ 123
$  60 ████████ 2,365
$  65 ████████████████████████████ 8,231
$  95 ████ 1,262
$ 115 ███████████████████████████████████████ 11,372
$ 165 █ 30
$ 180 █ 18


Conclusion 

We find, using both easy and hard methods, that the typical ticket costs $50: this is both the median and the most common fine. The average ticket amount (about $67.55) is higher because tickets of the amount $115 are nearly as common as tickets of the $50 amount. Importantly, these are the first 50,000 tickets from the API, and not a random sample; also, tickets with no fine amount were removed. 